In [1]:
import torch
import train

# Checking models performance on small number of epochs

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Sample DataFrame
data = {
    'Category': ['A', 'B', 'C'],
    'Metric1': [10, 20, 15],
    'Metric2': [5, 25, 10],
    'Metric3': [8, 12, 20]
}
df = pd.DataFrame(data)

# Create subplot grid
fig = make_subplots(rows=3, cols=1, subplot_titles=('Metric 1', 'Metric 2', 'Metric 3'))

# Store visibility configurations and buttons
visibilities = []
buttons = []

# Add traces for each row
for idx, row in df.iterrows():
    traces = []

    # Create a trace for each metric column
    for row_idx, col in enumerate(['Metric1', 'Metric2', 'Metric3']):
        trace = go.Bar(
            x=[row['Category']],
            y=[row[col]],
            name=f"{row['Category']} - {col}",
            visible=(idx == 0),  # Only first row visible initially
            showlegend=False
        )
        fig.add_trace(trace, row=row_idx + 1, col=1)
        traces.append(trace)
    
    # Create visibility array for this row (True for current row, False for others)
    visibility = [t.visible for t in fig.data]
        
        # Create dropdown button
    buttons.append({
        'label': row['Category'],
        'method': 'update',
        'args': [
            {'visible': [visibility]},
            {'title': f'Showing data for {row["Category"]}'}
        ]
    })

buttons = [
    {
        'label': df.iloc[i][0],
        'method': 'update',
        'args': [
            {'visible': [True if i * 3 <= j < (i + 1) * 3 else False for j in range(df.shape[0] * df.shape[1])]},
            {'title': f'Showing data for {row["Category"]}'}
        ]
    }
    for i in range(df.shape[0])
]

# Update layout with dropdown
fig.update_layout(
    updatemenus=[{
        'buttons': buttons,
        'direction': 'down',
        'showactive': True,
        'x': 0.5,
        'xanchor': 'center',
        'y': 1.15,
        'yanchor': 'top'
    }],
    title='Interactive DataFrame Visualization',
    height=1500
)

# Show figure
fig.show()

C:\Users\mokrota\AppData\Local\Temp\ipykernel_92964\889833574.py:52: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [ ]:
def traces_losses(losses_list):
    traces = []
    for j, loss in enumerate(losses_list):
            x = list(range(len(loss)))
            traces.append(go.Scatter(x=x, y=loss, mode='lines', name=f'Model {j}'))
    return traces

def traces_label_accuracy(label_accuracies):
      traces = []
      for i, 

In [46]:
losses_list = [list(range(i, 0, -1)) for i in range(10)]

fig = go.Figure()
fig.add_traces(traces_losses(losses_list))
fig.show()

## Round 1

In [47]:
import os, json
import plotly.graph_objects as go
def open_selected(prefix, suffix, index_list=None, path=train.save_path):
    l = []
    path = os.path.join(path, 'tmp')

    if index_list is None:
        i = 0
        path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
        while os.path.exists(path):
            with open(path, 'r') as f:
                data = json.load(f)
            l.append(data)

            i += 1
            path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
    else:
        for i in index_list:
            path = os.path.join(os.path.dirname(path), prefix + str(i) + suffix)
            with open(path, 'r') as f:
                data = json.load(f)
            l.append(data)
    return l

In [51]:
def visualise_losses(index_list=None, path=train.save_path):
    losses_list = open_selected(prefix="loss", suffix=".json", index_list=index_list, path=train.save_path)
    if index_list is None:
        index_list = list(range(losses_list))
    
    fig = go.Figure()
    for i, loss in zip(index_list, losses_list):
        x = list(range(len(loss)))
        fig.add_trace(go.Scatter(x=x, y=loss, mode="lines", name=f"Model {i}"))
    
    fig.update_layout(
        title="Cross Entropy Loss Of Models",
        xaxis_title="Batch",
        yaxis_title="Cross Entropy Loss",
        template="plotly_dark"
    )
    fig.show()

In [83]:
from plotly.subplots import make_subplots
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
def visualise_metrics(index_list=None, path=train.save_path):
    vals = open_selected(prefix="validation", suffix=".json", index_list=index_list, path=path)
    if index_list is None:
        index_list = list(range(len(vals)))
    labels = [f"Model {i}" for i in index_list]

    fig = make_subplots(rows=2, cols=3, subplot_titles=["Accuracy", "F1-Score Micro", "F1-Score Macro", "F1-Score Min", "F1-Score Max"])

    accuracies = [accuracy_score(i["expected"], i["predicted"]) for i in vals]
    f1_micros = [f1_score(i["expected"], i["predicted"], average="micro") for i in vals]
    f1_macros = [f1_score(i["expected"], i["predicted"], average="macro") for i in vals]
    f1_mins = []
    f1_maxs = []
    for val in vals:
        predicted = val['predicted']
        expected = val['expected']

        f1 = f1_score(predicted, expected, average=None)
        f1_min = min(f1)
        f1_max = max(f1)

        f1_mins.append(f1_min)
        f1_maxs.append(f1_max)

    fig.add_trace(go.Bar(
        x=labels,
        y=accuracies, name="Accuracy"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_micros, name="F1 Micro"
    ), row=1, col=2)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_macros, name="F1 Macro"
    ), row=1, col=3)

    fig.add_trace(go.Bar(
        x=labels,
        y=f1_mins, name="Min F1-Score Across Labels"
    ), row=2, col=1)
    fig.add_trace(go.Bar(
        x=labels,
        y=f1_maxs, name="Max F1-Score Across Labels"
    ), row=2, col=2)
    
    y_range = [0.0, 1.0]
    fig.update_xaxes(title_text="Models", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", row=1, col=1, range=y_range)
    fig.update_xaxes(title_text="Models", row=1, col=2)
    fig.update_yaxes(title_text="F1-Micro", row=1, col=2, range=y_range)
    fig.update_xaxes(title_text="Models", row=1, col=3)
    fig.update_yaxes(title_text="F1-Macro", row=1, col=3, range=y_range)

    fig.update_xaxes(title_text="Models", row=2, col=1)
    fig.update_yaxes(title_text="F1-Min", row=2, col=1, range=y_range)
    fig.update_xaxes(title_text="Models", row=2, col=2)
    fig.update_yaxes(title_text="F1-Max", row=2, col=2, range=y_range)

    fig.update_layout(
        title="Metrics Comparison Of Models",
        template="plotly_dark",
        height=1000
    )
    fig.show()

In [85]:
from sklearn.metrics import confusion_matrix
def visualise_label_accuracies(index_list=None, path=train.save_path):
    vals = open_selected(prefix="validation", suffix=".json", index_list=index_list, path=path)
    if index_list is None:
        index_list = list(range(len(vals)))
    models_no = [f"Model {i}" for i in index_list]
    labels = [chr(ord('a') + i) for i in range(26)]

    cms = [confusion_matrix(val['expected'], val['predicted']) for val in vals]
    label_accs = [cm.diagonal() / cm.sum(axis=1) for cm in cms]
    
    fig = go.Figure()

    for i, acc in enumerate(label_accs):
        fig.add_trace(go.Bar(
            x=labels,
            y=acc,
            name=f"Model {models_no[i]}",
            visible=(i==0)
        ))
    buttons = [
        {
            "label": f"Matrix {i}",
            "method": "update",
            "args": [{"visible": [j == i for j in range(len(label_accs))]}]
        }
        for i in range(len(label_accs))
    ]
    
    fig.update_layout(
        title="Accuracies Per Label",
        xaxis_title="Label",
        yaxis_title="Accuracy",
        updatemenus=[{
        'buttons': buttons,
        'direction': 'down',
        'showactive': True,
        'xanchor': 'center',
        'yanchor': 'top'
        }],
        template="plotly_dark"
    )
    fig.update_yaxes(range=[0.0, 1.0])
    fig.show()

In [6]:
seed = 123456789
batchsize = 32

In [7]:
n = 10
c_low = 5
c_high = 15
num_l = 2
exp_low = -13
exp_high = -4
epochs = 2

models, alphas = train.models_rand(n, c_low, c_high, exp_low, exp_high, num_l=num_l)
hyperparams_list = [{"lr": a} for a in alphas]

for i, (model, hyperparams) in enumerate(zip(models, hyperparams_list)):
    print(f"Training model {i}; channels: {model.summary}; hyperparams: {hyperparams}")
    train.train(model, batch_size=batchsize, hyperparams=hyperparams, epochs=epochs, seed=seed)

Training model 0; channels: [8, 7]; hyperparams: {'lr': 0.00048828125}


c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv2d(
c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch 1, batch 100, loss 3.256
Epoch 1, batch 200, loss 3.213
Epoch 1, batch 300, loss 3.137
Epoch 1, batch 400, loss 3.079
Epoch 1, batch 500, loss 3.053
Epoch 1, batch 600, loss 3.014
Epoch 1, batch 700, loss 3.012
Epoch 1, batch 800, loss 2.984
Epoch 1, batch 900, loss 2.974
Epoch 1, batch 1000, loss 2.994
Epoch 1, batch 1100, loss 2.974
Epoch 1, batch 1200, loss 2.951
Epoch 1, batch 1300, loss 2.949
Epoch 1, batch 1400, loss 2.962
Epoch 1, batch 1500, loss 2.949
Epoch 1, batch 1600, loss 2.949
Epoch 1, batch 1700, loss 2.925
Epoch 1, batch 1800, loss 2.924
Epoch 1, batch 1900, loss 2.919
Epoch 1, batch 2000, loss 2.930
Epoch 1, batch 2100, loss 2.904
Epoch 1, batch 2200, loss 2.882
Epoch 1, batch 2300, loss 2.883
Epoch 1, batch 2400, loss 2.877
Epoch 1, batch 2500, loss 2.862
Epoch 1, batch 2600, loss 2.869
Epoch 1, batch 2700, loss 2.864
Epoch 1, batch 2800, loss 2.859
Epoch 1, batch 2900, loss 2.842
Epoch 1, batch 3000, loss 2.844
Epoch 1, batch 3100, loss 2.861
Epoch 2, batch 10

In [69]:
visualise_losses(list(range(10)))

As we can see data variance of loss is too big, it is not steadily converging. To resolve this let's increase batch size

## Round 2

In [9]:
seed = 123456789
batch_size = 128

In [10]:
models, alphas = train.models_rand(n, c_low, c_high, exp_low, exp_high)
hyperparams_list = [{"lr": a} for a in alphas]

for i, (model, hyperparams) in enumerate(zip(models, hyperparams_list)):
    print(f"Training model {i}; channels: {model.summary}; hyperparams: {hyperparams}")
    train.train(model, batch_size=batch_size, hyperparams=hyperparams, epochs=epochs, seed=seed)

Training model 0; channels: [12, 5]; hyperparams: {'lr': 0.0001220703125}


c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



Epoch 1, batch 100, loss 3.258
Epoch 1, batch 200, loss 3.243
Epoch 1, batch 300, loss 3.162
Epoch 1, batch 400, loss 3.081
Epoch 1, batch 500, loss 3.030
Epoch 1, batch 600, loss 2.987
Epoch 1, batch 700, loss 2.960
Epoch 2, batch 100, loss 2.920
Epoch 2, batch 200, loss 2.908
Epoch 2, batch 300, loss 2.896
Epoch 2, batch 400, loss 2.874
Epoch 2, batch 500, loss 2.872
Epoch 2, batch 600, loss 2.860
Epoch 2, batch 700, loss 2.860
Training model 1; channels: [11, 10]; hyperparams: {'lr': 0.0009765625}
Epoch 1, batch 100, loss 3.207
Epoch 1, batch 200, loss 3.046
Epoch 1, batch 300, loss 2.949
Epoch 1, batch 400, loss 2.888
Epoch 1, batch 500, loss 2.841
Epoch 1, batch 600, loss 2.819
Epoch 1, batch 700, loss 2.798
Epoch 2, batch 100, loss 2.763
Epoch 2, batch 200, loss 2.747
Epoch 2, batch 300, loss 2.732
Epoch 2, batch 400, loss 2.721
Epoch 2, batch 500, loss 2.714
Epoch 2, batch 600, loss 2.701
Epoch 2, batch 700, loss 2.697
Training model 2; channels: [13, 6]; hyperparams: {'lr': 0.0

In [77]:
visualise_losses(list(range(10, 20)))

This is much better, although it is still noisy increasing batch size would hinder speed of computation so we've decided to stick with this

Now let's check performance via confusion matrix

In [12]:
# load if necessary
def load_model(folder, model_i):
    metadata_path = os.path.dirname(train.save_path)
    model_path = os.path.join(metadata_path, folder, f'cnn{model_i}.pth')
    metadata_path = os.path.join(metadata_path, folder, f"train_metadata{model_i}.json")
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    model = train.ConvNetPooling(train.height, train.width, train.output_size, channels=metadata['channels'])
    model.load_state_dict(torch.load(model_path, weights_only=True))
    return model

# folder = "training_pyramid3"
# models = [load_model(folder, i) for i in range(5, 10)]

In [86]:
visualise_label_accuracies(list(range(n, 2 * n)))

In [87]:
visualise_metrics(list(range(n, 2 * n)))

# Training

Now that we've decided to stick with model 3 let's train it for more epochs

In [15]:
no = 13
hyperparams = {'lr': 0.0009765625}
model = load_model("training_pyramid16", str(no))

In [16]:
seed = 123456789
batch_size = 128

In [17]:
epochs = 20
train.train(model=model, hyperparams=hyperparams, batch_size=batch_size, epochs=epochs)

c:\Users\mokrota\Documents\GitHub\Neural-Network-Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning:

Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.



Epoch 1, batch 100, loss 2.655
Epoch 1, batch 200, loss 2.657
Epoch 1, batch 300, loss 2.653
Epoch 1, batch 400, loss 2.645
Epoch 1, batch 500, loss 2.639
Epoch 1, batch 600, loss 2.629
Epoch 1, batch 700, loss 2.624
Epoch 2, batch 100, loss 2.615
Epoch 2, batch 200, loss 2.618
Epoch 2, batch 300, loss 2.622
Epoch 2, batch 400, loss 2.618
Epoch 2, batch 500, loss 2.614
Epoch 2, batch 600, loss 2.612
Epoch 2, batch 700, loss 2.607
Epoch 3, batch 100, loss 2.593
Epoch 3, batch 200, loss 2.595
Epoch 3, batch 300, loss 2.600
Epoch 3, batch 400, loss 2.597
Epoch 3, batch 500, loss 2.594
Epoch 3, batch 600, loss 2.595
Epoch 3, batch 700, loss 2.586
Epoch 4, batch 100, loss 2.589
Epoch 4, batch 200, loss 2.578
Epoch 4, batch 300, loss 2.576
Epoch 4, batch 400, loss 2.584
Epoch 4, batch 500, loss 2.579
Epoch 4, batch 600, loss 2.579
Epoch 4, batch 700, loss 2.574
Epoch 5, batch 100, loss 2.567
Epoch 5, batch 200, loss 2.576
Epoch 5, batch 300, loss 2.582
Epoch 5, batch 400, loss 2.567
Epoch 5,

In [80]:
visualise_losses([20])

In [88]:
visualise_label_accuracies([20])
visualise_metrics([20])